# Multi-energy phase retrieval with refractive-index constraints

This notebook jointly reconstructs a stack of holograms using the explicit

$$L_E(r) = C(r) + M(r)a_E$$

model. Prepared $\beta(E)$ and $\delta(E)$ arrays constrain the absorptive and dispersive spectral dependence. Henke extension and KK calculation are performed beforehand, for example with `01_absorption_to_refractive_index.ipynb`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from library import phase_retrieval_core_multienergy as phr_me

## Load prepared holograms and constraints

`multi_energy_inputs.npz` should contain `holograms` with shape `(nE, nx, ny)`, `energy_ev`, `mask_pixel`, and `supportmask`. `mask_pixel` may be shared `(nx, ny)` or energy dependent `(nE, nx, ny)`.

In [ ]:
data = np.load(Path("data/multi_energy_inputs.npz"))
holograms = data["holograms"]
energy_ev = data["energy_ev"]
mask_pixel = data["mask_pixel"]
supportmask = data["supportmask"]

optical_constants = np.load(Path("data/refractive_index_constraints.npz"))
constraint_energy_ev = optical_constants["energy_ev"]
beta = optical_constants["beta"]
delta = optical_constants["delta"]

if not np.allclose(energy_ev, constraint_energy_ev):
    beta = np.interp(energy_ev, constraint_energy_ev, beta)
    delta = np.interp(energy_ev, constraint_energy_ev, delta)

assert holograms.shape[0] == energy_ev.size == beta.size == delta.size
assert supportmask.shape == holograms.shape[1:]

## Joint reconstruction

`known_beta_kk` uses the supplied $\beta$. Because `known_delta_spectrum` is also supplied, no KK integration or Henke access occurs inside phase retrieval. `absorption_part="real"` matches the log-object convention used by this library.

In [ ]:
recipe = {
    "mode": "HAPRE",
    "outer_iterations": 300,
    "inner_iterations": 1,
    "warmup_iterations": 20,
    "shuffle_energies": True,
    "random_seed": 0,
    "beta_zero": 0.5,
    "beta_mode": "arctan",
    "plot_every": 1e9,
    "average_img": 1,
    "projection_model": "rank1_spectral",
    "projection_every": 1,
    "projection_start": 0,
    "projection_relaxation": 1.0,
    "projection_static_mode": "mean",
    "spectral_constraint": "known_beta_kk",
    "energy_values": energy_ev,
    "known_beta_spectrum": beta,
    "known_delta_spectrum": delta,
    "absorption_part": "real",
    "kk_sign": 1.0,
    "known_beta_normalization": "none",
    "fit_known_beta_scale": True,
    "fit_known_beta_offset": False,
    "final_fourier_constraint": True,
}

## Same-energy opposite-helicity pair reconstructions

Before coupling the full stack across energy, run independent two-helicity reconstructions for matching-energy hologram pairs when both helicity stacks are available in `multi_energy_inputs.npz`.


In [ ]:
# Expected optional arrays: one stack for each helicity, shape (nE, ny, nx).
positive_helicity_keys = ("pos_holograms", "positive_holograms", "cr_holograms", "CR")
negative_helicity_keys = ("neg_holograms", "negative_holograms", "cl_holograms", "CL")

pos_key = next((key for key in positive_helicity_keys if key in data.files), None)
neg_key = next((key for key in negative_helicity_keys if key in data.files), None)

same_energy_pair_results = {}
if pos_key is None or neg_key is None:
    print(
        "Skipping same-energy pair retrieval: "
        "multi_energy_inputs.npz does not contain both helicity stacks."
    )
else:
    pos_holograms = np.asarray(data[pos_key])
    neg_holograms = np.asarray(data[neg_key])
    if pos_holograms.shape != neg_holograms.shape:
        raise ValueError(
            f"Opposite-helicity stacks differ in shape: "
            f"{pos_holograms.shape} vs {neg_holograms.shape}"
        )
    if pos_holograms.shape[0] != energy_ev.size:
        raise ValueError("The helicity-pair energy axis does not match energy_ev")

    pair_recipe = {
        "algorithm_list": ["HAPRE", "ER", "HAPRE", "ER"],
        "number_iterations": [700, 50, 700, 50],
        "helicity": ["pos", "pos", "neg", "neg"],
        "beta_zero": [0.5, 0.5, 0.5, 0.5],
        "beta_mode": ["arctan", "const", "arctan", "const"],
        "alpha_zero": [0.0, 0.0, 0.0, 0.0],
        "alpha_mode": ["const", "const", "const", "const"],
        "RL_its": [0, 0, 0, 0],
        "RL_freqs": [1e9, 1e9, 1e9, 1e9],
        "TV_freqs": [1e9, 1e9, 1e9, 1e9],
        "plot_every": [350, 25, 350, 25],
        "average_img": [30, 30, 30, 30],
        "Fourier_last": [True, True, True, True],
        "hologram_intensity_cutoff_vmin": -1,
        "Startimage": [None, "pos", "pos", "neg"],
        "Startgamma": [None, None, None, None],
    }

    pair_indices = range(energy_ev.size)
    for energy_index in pair_indices:
        pair_mask = mask_pixel[energy_index] if mask_pixel.ndim == 3 else mask_pixel
        results = phr_me.phase_retrieval_algorithm(
            pos_holograms[energy_index],
            neg_holograms[energy_index],
            pair_mask,
            supportmask,
            phase_retrieval_recipe=pair_recipe,
        )
        same_energy_pair_results[float(energy_ev[energy_index])] = results
        print(f"retrieved opposite-helicity pair at {energy_ev[energy_index]:.1f} eV")


In [ ]:
retrieved, components, bsmasks, error = (
    phr_me.multi_energy_phase_retrieval_algorithm(
        holograms,
        mask_pixel,
        supportmask,
        multi_energy_recipe=recipe,
    )
)

print(f"Runtime: {error['runtime_seconds']} s")
print("Retrieved stack:", retrieved.shape)

## Inspect the result

In [ ]:
log_objects = phr_me.fourier_field_to_object_log(retrieved)

fig, axes = plt.subplots(2, energy_ev.size, figsize=(3 * energy_ev.size, 6))
for index, energy in enumerate(energy_ev):
    axes[0, index].imshow(np.real(log_objects[index]), cmap="magma")
    axes[0, index].set_title(f"{energy:.1f} eV\nabsorption-like")
    axes[1, index].imshow(np.imag(log_objects[index]), cmap="twilight")
    axes[1, index].set_title("phase-like")
for ax in axes.ravel():
    ax.axis("off")
plt.show()

fig, ax = plt.subplots()
ax.plot(energy_ev, beta, "o-", label=r"input $\beta$")
ax.plot(energy_ev, delta, "o-", label=r"input $\delta$")
ax.set_xlabel("Photon energy (eV)")
ax.grid(True)
ax.legend()
plt.show()